# HTG Overview: Basenodes, Multi-Node Graphs, and Hierarchical Composition

The **Hierarchal Tensor Graph** (HTG) is the core object in CREST where an Earth System
Model (ESM) is encoded. It provides the structure needed to build a CREST model using a
Tensor Network backend. As its name suggests, an HTG is hierarchical: nodes within an HTG
can themselves be HTGs.

This notebook covers:

1. **Basenodes** — single-node HTGs wrapping a Python callable
2. **Multi-node HTGs** — composing basenodes into coupled sub-systems
3. **I/O renaming** — aligning key names across edges with `rename`
4. **HTG as class** — encapsulating graph construction in a reusable `HierarchalTensorGraph` subclass
5. **Hierarchical nesting** — using one HTG as a node inside another
6. **Node access and iteration** — indexing, `all_nodes`, `__iter__`, and `__contains__`

In [ ]:
# Allow crest to be imported
%cd -q ..

In [ ]:
from crest import HierarchalTensorGraph, Node

## 1. HTG Basenodes

An HTG has two modes:

1. **Basenode** (single-node graph) — wraps a Python callable to represent a fundamental
   physical process. Created with the `Node` class, which requires a `node` callable plus
   `inputs` and `outputs` dicts describing the tensor interface.

2. **Multi-node graph** — represents a coupled sub-system. Created by instantiating
   `HierarchalTensorGraph` directly (no callable) and adding nodes/edges. Each node can
   itself be a basenode or another multi-node HTG, enabling arbitrary depth of hierarchy.

#### A single node graph

In [ ]:
process = lambda X : {'square' : X['x']*X['x']}
sq = Node(node=process, inputs={'x': None}, outputs={'square': None})

In [ ]:
sq.name

In [ ]:
# Name defaults to the callable's __name__ or __qualname__.
# Best practice: always set it explicitly at construction or right after:
sq.name = 'square'

In [ ]:
if sq.is_basenode:
    print(f'{sq.name} is a basenode (no internal graph)')

In [ ]:
X = {'x' : 2}
sq(X)

## 2. Multi-Node HTGs

Multi-node HTGs are used to represent coupled sub-systems. Nodes are added implicitly
when you call `add_edge` — there is no separate `add_node` step. Nodes can be basenodes
or other multi-node HTGs.

Below we build `(x_1 + x_2)^2` by connecting an `add` node to the `square` basenode
we defined above. The output key `"sum"` from `add` must be renamed to `"x"` to match
the input key that `square` expects.

In [ ]:
process = lambda X : {'sum' : X['x_1'] + X['x_2']}
add = Node(node=process, inputs={'x_1': None, 'x_2': None}, outputs={'sum': None}, name='add')
add_square = HierarchalTensorGraph(name='add_square')

add_square.add_edge('input', add)
add_square.add_edge(sq, 'output')
add_square.add_edge(add, sq, rename={'sum': 'x'})
add_square.draw()

In [ ]:
f'__getitem__ = {add_square["add"]}'

In [ ]:
# Get a dictionary of the nodes in the HTG
add_square.nodes

## 3. Renaming I/O in HTGs

Output keys produced by one node often differ from the input keys expected by the next.
The `rename` parameter in `add_edge` maps `{source_output_key: target_input_key}` to
bridge this mismatch — no changes to the node definitions are needed.

Extra keys passed by a caller that no node consumes are silently ignored.

In [ ]:
print(f"add inputs:    {add_square['add'].inputs}")
print(f"add outputs:   {add_square['add'].outputs}")
print(f"square inputs:  {add_square['square'].inputs}")
print(f"square outputs: {add_square['square'].outputs}")

In [ ]:
add_square.print_edge_labels()

In [ ]:
X = {'x_1' : 5, 'x_2' : 3}
add_square(X)

In [ ]:
print(f"add_square inputs:  {add_square.inputs}")
print(f"add_square outputs: {add_square.outputs}")

In [ ]:
X = {'x_1' : 5, 'x_2' : 3, 'z' : 12}
add_square(X)

In [ ]:
# Sometimes we may want to wrap our HTG in a class for more flexibility
class AddSquare(HierarchalTensorGraph):

    def __init__(self, name='add_square'):
        super().__init__(name=name)

        # Create an add basenode with its I/O spec
        add = Node(
            node=lambda X : {'sum' : X['x_1'] + X['x_2']},
            inputs={'x_1': None, 'x_2': None},
            outputs={'sum': None},
            name='add'
        )

        # Create a square basenode with its I/O spec
        square = Node(
            node=lambda X : {'square' : X['x']*X['x']},
            inputs={'x': None},
            outputs={'square': None},
            name='square'
        )

        # Build the graph; rename 'sum' -> 'x' on the add->square edge,
        # and 'square' -> 'add_square' on the square->output edge
        self.add_edge('input', add)
        self.add_edge(add, square, rename={'sum': 'x'})
        self.add_edge(square, 'output', rename={'square': 'add_square'})

## 4. HTG as a Reusable Class

Wrapping an HTG in a Python class (by subclassing `HierarchalTensorGraph`) lets you
parameterize construction, reuse the HTG in multiple graphs, and keep the graph-building
logic encapsulated. The `__init__` method calls `super().__init__(name=name)` and then
builds the graph with `add_edge` calls.

In [ ]:
# Evaluate our class
X = {'x_1' : 5, 'x_2' : 3, 'z' : 12}
add_sq = AddSquare()
add_sq(X)

## 5. Adding Another Layer: $log((x_1 + x_2)^2)$

An HTG class can use another HTG class as a child node — just instantiate it and pass it
to `add_edge`. CREST treats it identically to a basenode. The `draw()` call renders each
child HTG as a collapsed box; use `expand_nodes` to unfold it.

In [ ]:
from math import log

class LogAddSquare(HierarchalTensorGraph):

    def __init__(self, name='log_add_square'):
        super().__init__(name=name)

        lg = Node(
            node=lambda X : {'log' : log(X['x'])},
            inputs={'x': None},
            outputs={'log': None},
            name='log'
        )

        self.add_edge('input', AddSquare())
        self.add_edge('add_square', lg, rename={'add_square': 'x'})
        self.add_edge('log', 'output', rename={'log': name})

In [ ]:
las = LogAddSquare()
las(X)

In [ ]:
las.draw()

In [ ]:
las.draw(expand_nodes='add_square')

In [ ]:
las.nodes

## 6. Node Access and Iteration

CREST provides several ways to inspect and traverse an HTG's nodes:

- `htg.nodes` — dict of the **direct** children of this HTG (one level deep)
- `htg.all_nodes` — dict of **all** nodes in the entire hierarchy, keyed by tuples of names
- `htg['name']` / `htg[('name',)]` — access a direct child by name
- `htg[('parent', 'child')]` — access nodes deeper in the hierarchy (tuple key)
- `for path, node in htg` — iterate over all nodes with their full path tuple
- `'name' in htg` / `('parent', 'child') in htg` — membership test
- `node.output` — the output dict produced by the most recent call to that node

In [ ]:
las.all_nodes

In [ ]:
las['log']

In [ ]:
las[('log')]

In [ ]:
las[('add_square','add')]

In [ ]:
for path, node in las:
    print(path, '->', node.name)

In [ ]:
print([
    'log' in las, 
    ('log') in las, 
    ('add_square','add') in las,
    'add' in las]
)

In [ ]:
print([las['log'].output, las[('add_square','add')].output])

In [ ]:
las.print_edge_labels(expand_nodes='all')